In [ ]:
!pip -q install -U langchain langchain-openai langchain-core openai
!pip -q install pandas==2.2.2 "scikit-learn>=1.2,<1.9"

In [ ]:
# =========================================================
# 0. Install dependencies in Colab
# =========================================================
# Run this cell first if needed:
# !pip -q install pandas==2.2.2 "scikit-learn>=1.2,<1.9"
# !pip -q install requests python-docx jinja2 statsmodels openai
# !pip -q install langchain langchain-openai


# =========================================================
# 1. API key for Colab
# =========================================================

import os

# Do not commit the real key to GitHub.
# Recommended: set OPENAI_API_KEY in Colab secrets or environment variables.
OPENAI_API_KEY = "key"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# =========================================================
# 2. Imports
# =========================================================

import json
import time
import re
import math
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd

from pydantic import BaseModel, Field

from sklearn.base import clone
from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression,
    RidgeClassifier,
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI


# =========================================================
# 3. General settings
# =========================================================

LLM_PROVIDER = "OpenAI"
OPENAI_MODEL = "gpt-5.5"
OPENAI_MODEL_DISPLAY_NAME = "GPT-5.5"

# GPT-5.5 experiment:

DISABLE_SAMPLING_PARAMS = True
TEMPERATURE_SENT = False
TEMPERATURE_REQUESTED = None

DATA_DIR = "/content"
SAVE_MESSAGE_TRACE = False


# =========================================================
# 4. Choose one config by commenting / uncommenting
# =========================================================

# -------------------------
# Regression configs
# -------------------------
CONFIG_PATH = "/content/regression_linear_regression_experiment_config.json"
# CONFIG_PATH = "/content/regression_random_forest_regressor_experiment_config.json"
# CONFIG_PATH = "/content/regression_gradient_boosting_regressor_experiment_config.json"

# -------------------------
# Binary classification configs
# -------------------------
# CONFIG_PATH = "/content/binary_logistic_regression_experiment_config.json"
# CONFIG_PATH = "/content/binary_lda_experiment_config.json"
# CONFIG_PATH = "/content/binary_ridge_classifier_experiment_config.json"

# -------------------------
# Multiclass classification configs
# -------------------------
# CONFIG_PATH = "/content/multiclass_random_forest_classifier_experiment_config.json"
# CONFIG_PATH = "/content/multiclass_decision_tree_classifier_experiment_config.json"
# CONFIG_PATH = "/content/multiclass_lda_experiment_config.json"


CONFIG_STEM = re.sub(
    r"[^a-zA-Z0-9_\-]+",
    "_",
    os.path.splitext(os.path.basename(CONFIG_PATH))[0]
)

OUTPUT_DIR = "/content/langchain_lin_outputs_gpt55"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# 5. Config loading and general helpers
# =========================================================

def load_experiment_configs(config_path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        configs = json.load(f)

    if not isinstance(configs, list):
        raise ValueError("The config JSON must contain a list of experiment configs.")

    return configs


def safe_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(name).strip())


def get_dataset_path(config: Dict[str, Any]) -> str:
    if config.get("dataset_path"):
        return config["dataset_path"]
    return os.path.join(DATA_DIR, config["file_name"])


def get_model_name(config: Dict[str, Any]) -> str:
    model_name = (
        config.get("configured_model_name")
        or config.get("model_name")
        or config.get("configured_model")
    )

    if not model_name:
        raise ValueError(f"Missing model name in config: {config.get('dataset_name')}")

    return model_name


def infer_task_type(config: Dict[str, Any]) -> str:
    if config.get("task_type"):
        return config["task_type"]

    model_name = get_model_name(config)

    if model_name in [
        "linear_regression",
        "random_forest_regressor",
        "gradient_boosting_regressor",
    ]:
        return "regression"

    if model_name in [
        "logistic_regression",
        "lda_classifier",
        "ridge_classifier",
    ]:
        return "binary_classification"

    if model_name in [
        "decision_tree_classifier",
        "random_forest_classifier",
    ]:
        return "multiclass_classification"

    raise ValueError(f"Unsupported model name: {model_name}")


MODEL_DISPLAY_NAMES = {
    "linear_regression": "Scikit-learn Linear Regression",
    "random_forest_regressor": "Random Forest Regression",
    "gradient_boosting_regressor": "Gradient Boosting Regression",
    "logistic_regression": "Logistic Regression",
    "lda_classifier": "Linear Discriminant Analysis",
    "ridge_classifier": "Ridge Classifier",
    "decision_tree_classifier": "Decision Tree Classifier",
    "random_forest_classifier": "Random Forest Classifier",
}


def get_sampling_control_metadata() -> Dict[str, Any]:
    return {
        "sampling_control": {
            "temperature_supported": False,
            "temperature_sent": TEMPERATURE_SENT,
            "temperature_requested": TEMPERATURE_REQUESTED,
            "top_p_sent": False,
            "presence_penalty_sent": False,
            "frequency_penalty_sent": False,
            "sampling_control_note": (
                "Sampling parameters are intentionally not sent for the GPT-5.5 experiment."
            ),
        }
    }


def safe_float(value: Any) -> Optional[float]:
    try:
        value = float(value)
        if math.isnan(value) or math.isinf(value):
            return None
        return value
    except Exception:
        return None


def make_json_safe(obj: Any) -> Any:
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")

    if isinstance(obj, pd.Series):
        return obj.to_dict()

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return safe_float(obj)

    if isinstance(obj, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]

    try:
        json.dumps(obj, ensure_ascii=False)
        return obj
    except Exception:
        return str(obj)


# =========================================================
# 6. Dataset and preprocessing helpers
# =========================================================

def load_dataset(config: Dict[str, Any]) -> pd.DataFrame:
    dataset_path = get_dataset_path(config)
    sep = config.get("csv_sep", ",")

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    return pd.read_csv(dataset_path, sep=sep)


def validate_columns(
    df: pd.DataFrame,
    x_columns: List[str],
    y_column: str,
    dataset_name: str
) -> None:
    missing = [c for c in x_columns + [y_column] if c not in df.columns]

    if missing:
        raise ValueError(
            f"[{dataset_name}] Missing columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )


def split_feature_types(df: pd.DataFrame, x_columns: List[str]) -> Dict[str, List[str]]:
    numeric_columns = []
    categorical_columns = []

    for col in x_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_columns.append(col)
        else:
            categorical_columns.append(col)

    return {
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
    }


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_unified_preprocessor(
    X: pd.DataFrame,
    x_columns: List[str],
) -> Tuple[ColumnTransformer, Dict[str, List[str]]]:
    feature_types = split_feature_types(X, x_columns)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, feature_types["numeric_columns"]),
            ("cat", categorical_transformer, feature_types["categorical_columns"]),
        ],
        remainder="drop",
    )

    return preprocessor, feature_types


def clean_preprocessed_feature_name(name: str) -> str:
    name = str(name)

    if name.startswith("num__"):
        return name.replace("num__", "", 1)

    if name.startswith("cat__"):
        return name.replace("cat__", "", 1)

    return name


def get_original_feature_from_raw_name(
    raw_name: str,
    numeric_columns: List[str],
    categorical_columns: List[str]
) -> str:
    raw_name = str(raw_name)

    if raw_name.startswith("num__"):
        return raw_name.replace("num__", "", 1)

    if raw_name.startswith("cat__"):
        rest = raw_name.replace("cat__", "", 1)

        for col in sorted(categorical_columns, key=len, reverse=True):
            if rest == col or rest.startswith(col + "_"):
                return col

        return rest.split("_")[0]

    return raw_name


def get_feature_names_from_pipeline(
    pipeline: Pipeline,
    feature_types: Dict[str, List[str]],
) -> Tuple[List[str], List[str]]:
    preprocessor = pipeline.named_steps["preprocessor"]
    raw_feature_names = list(preprocessor.get_feature_names_out())

    cleaned_feature_names = [
        clean_preprocessed_feature_name(name)
        for name in raw_feature_names
    ]

    return raw_feature_names, cleaned_feature_names


def detect_numeric_standardization(pipeline: Pipeline) -> bool:
    try:
        preprocessor = pipeline.named_steps["preprocessor"]

        for name, transformer, columns in preprocessor.transformers_:
            if name == "num":
                if hasattr(transformer, "named_steps"):
                    scaler = transformer.named_steps.get("scaler")
                    return isinstance(scaler, StandardScaler)

        return False

    except Exception:
        return False


def build_feature_interpretation_metadata(
    raw_feature_names: List[str],
    cleaned_feature_names: List[str],
    feature_types: Dict[str, List[str]],
    numeric_standardized: bool,
) -> List[Dict[str, Any]]:
    numeric_columns = set(feature_types.get("numeric_columns", []))
    categorical_columns = feature_types.get("categorical_columns", [])

    metadata = []

    for raw_name, cleaned_name in zip(raw_feature_names, cleaned_feature_names):
        original_feature = get_original_feature_from_raw_name(
            raw_name=raw_name,
            numeric_columns=feature_types.get("numeric_columns", []),
            categorical_columns=categorical_columns,
        )

        if original_feature in numeric_columns:
            feature_type = "numeric"

            if numeric_standardized:
                interpretation_unit = "one-standard-deviation increase after preprocessing"
            else:
                interpretation_unit = "one-unit increase on the original scale"

        else:
            feature_type = "categorical_or_onehot"
            interpretation_unit = "a change in the one-hot encoded category indicator from 0 to 1"

        metadata.append(
            {
                "raw_feature_name": raw_name,
                "preprocessed_feature_name": cleaned_name,
                "original_feature": original_feature,
                "feature_type": feature_type,
                "interpretation_unit": interpretation_unit,
            }
        )

    return metadata


def aggregate_effects_to_original_variables(
    raw_feature_names: List[str],
    cleaned_feature_names: List[str],
    effect_values: List[float],
    numeric_columns: List[str],
    categorical_columns: List[str],
) -> Dict[str, Any]:
    original_abs_effects = {}
    original_representative_effects = {}

    for raw_name, cleaned_name, effect_value in zip(
        raw_feature_names,
        cleaned_feature_names,
        effect_values,
    ):
        original_feature = get_original_feature_from_raw_name(
            raw_name=raw_name,
            numeric_columns=numeric_columns,
            categorical_columns=categorical_columns,
        )

        effect_value = float(effect_value)
        abs_effect = abs(effect_value)

        if (
            original_feature not in original_abs_effects
            or abs_effect > original_abs_effects[original_feature]
        ):
            original_abs_effects[original_feature] = abs_effect
            original_representative_effects[original_feature] = effect_value

    most_influential_original_feature = (
        max(original_abs_effects, key=original_abs_effects.get)
        if original_abs_effects
        else None
    )

    return {
        "original_feature_absolute_effects": {
            str(k): safe_float(v)
            for k, v in original_abs_effects.items()
        },
        "original_feature_representative_effects": {
            str(k): safe_float(v)
            for k, v in original_representative_effects.items()
        },
        "most_influential_original_feature": most_influential_original_feature,
        "most_influential_original_feature_value": (
            safe_float(original_abs_effects.get(most_influential_original_feature))
            if most_influential_original_feature
            else None
        ),
    }


# =========================================================
# 7. Model builders
# =========================================================

def build_regression_model(model_name: str):
    if model_name == "linear_regression":
        return LinearRegression()

    if model_name == "random_forest_regressor":
        return RandomForestRegressor(
            n_estimators=100,
            random_state=42,
        )

    if model_name == "gradient_boosting_regressor":
        return GradientBoostingRegressor(
            random_state=42,
        )

    raise ValueError(f"Unsupported regression model: {model_name}")


def build_classification_model(model_name: str):
    if model_name == "logistic_regression":
        return LogisticRegression(
            max_iter=2000,
            random_state=42,
        )

    if model_name == "lda_classifier":
        return LinearDiscriminantAnalysis()

    if model_name == "ridge_classifier":
        return RidgeClassifier(
            random_state=42,
        )

    if model_name == "decision_tree_classifier":
        return DecisionTreeClassifier(
            random_state=42,
        )

    if model_name == "random_forest_classifier":
        return RandomForestClassifier(
            n_estimators=100,
            random_state=42,
        )

    raise ValueError(f"Unsupported classification model: {model_name}")


def build_model(model_name: str, task_type: str):
    if task_type == "regression":
        return build_regression_model(model_name)

    if task_type in ["binary_classification", "multiclass_classification"]:
        return build_classification_model(model_name)

    raise ValueError(f"Unsupported task type: {task_type}")


def build_unified_pipeline(
    X: pd.DataFrame,
    x_columns: List[str],
    model_name: str,
    task_type: str,
) -> Tuple[Pipeline, Dict[str, List[str]]]:
    preprocessor, feature_types = build_unified_preprocessor(X, x_columns)
    model = build_model(model_name, task_type)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return pipeline, feature_types


# =========================================================
# 8. Metrics
# =========================================================

def safe_train_test_split_for_classification(
    X: pd.DataFrame,
    y: pd.Series
):
    try:
        counts = pd.Series(y).value_counts(dropna=False)
        if len(counts) > 1 and counts.min() >= 2:
            return train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
                stratify=y,
            )
    except Exception:
        pass

    return train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=None,
    )


def compute_train_test_regression_metrics(
    base_pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
) -> Dict[str, Any]:
    if len(X) < 5:
        return {
            "train_r2": None,
            "test_r2": None,
            "train_rmse": None,
            "test_rmse": None,
            "train_mae": None,
            "test_mae": None,
            "split_note": "Too few rows for a stable train/test split.",
        }

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
        )

        split_pipeline = clone(base_pipeline)
        split_pipeline.fit(X_train, y_train)

        train_pred = split_pipeline.predict(X_train)
        test_pred = split_pipeline.predict(X_test)

        return {
            "train_r2": safe_float(r2_score(y_train, train_pred)),
            "test_r2": safe_float(r2_score(y_test, test_pred)),
            "train_rmse": safe_float(mean_squared_error(y_train, train_pred, squared=False)),
            "test_rmse": safe_float(mean_squared_error(y_test, test_pred, squared=False)),
            "train_mae": safe_float(mean_absolute_error(y_train, train_pred)),
            "test_mae": safe_float(mean_absolute_error(y_test, test_pred)),
            "split_note": "Train/test split uses test_size=0.2 and random_state=42.",
        }

    except TypeError:
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
            )

            split_pipeline = clone(base_pipeline)
            split_pipeline.fit(X_train, y_train)

            train_pred = split_pipeline.predict(X_train)
            test_pred = split_pipeline.predict(X_test)

            train_mse = mean_squared_error(y_train, train_pred)
            test_mse = mean_squared_error(y_test, test_pred)

            return {
                "train_r2": safe_float(r2_score(y_train, train_pred)),
                "test_r2": safe_float(r2_score(y_test, test_pred)),
                "train_rmse": safe_float(math.sqrt(train_mse)),
                "test_rmse": safe_float(math.sqrt(test_mse)),
                "train_mae": safe_float(mean_absolute_error(y_train, train_pred)),
                "test_mae": safe_float(mean_absolute_error(y_test, test_pred)),
                "split_note": "Train/test split uses test_size=0.2 and random_state=42.",
            }

        except Exception as exc:
            return {
                "train_r2": None,
                "test_r2": None,
                "train_rmse": None,
                "test_rmse": None,
                "train_mae": None,
                "test_mae": None,
                "split_note": f"Train/test regression metric computation failed: {str(exc)}",
            }

    except Exception as exc:
        return {
            "train_r2": None,
            "test_r2": None,
            "train_rmse": None,
            "test_rmse": None,
            "train_mae": None,
            "test_mae": None,
            "split_note": f"Train/test regression metric computation failed: {str(exc)}",
        }


def compute_train_test_accuracy(
    base_pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
) -> Dict[str, Any]:
    if len(X) < 5:
        return {
            "train_accuracy": None,
            "test_accuracy": None,
            "split_note": "Too few rows for a stable train/test split.",
        }

    try:
        X_train, X_test, y_train, y_test = safe_train_test_split_for_classification(X, y)

        split_pipeline = clone(base_pipeline)
        split_pipeline.fit(X_train, y_train)

        train_pred = split_pipeline.predict(X_train)
        test_pred = split_pipeline.predict(X_test)

        return {
            "train_accuracy": safe_float(accuracy_score(y_train, train_pred)),
            "test_accuracy": safe_float(accuracy_score(y_test, test_pred)),
            "split_note": (
                "Train/test split uses test_size=0.2 and random_state=42; "
                "stratification is used when feasible."
            ),
        }

    except Exception as exc:
        return {
            "train_accuracy": None,
            "test_accuracy": None,
            "split_note": f"Train/test accuracy computation failed: {str(exc)}",
        }


# =========================================================
# 9. Regression evidence
# =========================================================

def try_compute_linear_regression_pvalues(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    feature_names: List[str],
) -> List[float]:
    try:
        import statsmodels.api as sm

        X_preprocessed = pipeline.named_steps["preprocessor"].transform(X)

        if hasattr(X_preprocessed, "toarray"):
            X_preprocessed = X_preprocessed.toarray()

        X_with_const = sm.add_constant(X_preprocessed, has_constant="add")
        model = sm.OLS(y.astype(float).to_numpy(), X_with_const).fit()

        pvalues = list(model.pvalues)

        if len(pvalues) == len(feature_names) + 1:
            return [
                safe_float(p) if safe_float(p) is not None else 1.0
                for p in pvalues[1:]
            ]

        if len(pvalues) >= len(feature_names):
            return [
                safe_float(p) if safe_float(p) is not None else 1.0
                for p in pvalues[-len(feature_names):]
            ]

    except Exception:
        pass

    return [1.0 for _ in feature_names]


def get_regression_effects(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
    raw_feature_names: List[str],
    cleaned_feature_names: List[str],
) -> Dict[str, Any]:
    trained_model = pipeline.named_steps["model"]
    n_features = len(cleaned_feature_names)

    if model_name == "linear_regression" and hasattr(trained_model, "coef_"):
        effect_values = np.asarray(trained_model.coef_, dtype=float).ravel()

        if len(effect_values) < n_features:
            effect_values = np.pad(effect_values, (0, n_features - len(effect_values)))
        elif len(effect_values) > n_features:
            effect_values = effect_values[:n_features]

        p_values = try_compute_linear_regression_pvalues(
            pipeline=pipeline,
            X=X,
            y=y,
            feature_names=cleaned_feature_names,
        )

        evidence_type = "coefficients"
        effect_note = (
            "The model exposes regression coefficients. Positive coefficients indicate positive associations "
            "with the dependent variable, while negative coefficients indicate negative associations. "
            "For standardized numeric predictors, coefficients are interpreted in terms of one-standard-deviation "
            "changes after preprocessing."
        )

    elif hasattr(trained_model, "feature_importances_"):
        effect_values = np.asarray(trained_model.feature_importances_, dtype=float).ravel()

        if len(effect_values) < n_features:
            effect_values = np.pad(effect_values, (0, n_features - len(effect_values)))
        elif len(effect_values) > n_features:
            effect_values = effect_values[:n_features]

        p_values = [None for _ in cleaned_feature_names]

        evidence_type = "feature_importances"
        effect_note = (
            "The model exposes feature importances. Feature importance indicates relative predictive strength "
            "but not positive or negative direction."
        )

    else:
        effect_values = np.zeros(n_features, dtype=float)
        p_values = [None for _ in cleaned_feature_names]

        evidence_type = "unavailable"
        effect_note = (
            "The configured regression model does not expose coefficients or feature importances."
        )

    effect_values = [float(v) for v in effect_values]

    effect_dict = {
        str(feature): safe_float(value)
        for feature, value in zip(cleaned_feature_names, effect_values)
    }

    p_value_dict = {
        str(feature): safe_float(value) if value is not None else None
        for feature, value in zip(cleaned_feature_names, p_values)
    }

    absolute_effect_dict = {
        str(feature): safe_float(abs(value))
        for feature, value in zip(cleaned_feature_names, effect_values)
    }

    most_influential_preprocessed_feature = (
        max(absolute_effect_dict, key=absolute_effect_dict.get)
        if absolute_effect_dict
        else None
    )

    top_10_preprocessed_features = sorted(
        absolute_effect_dict.items(),
        key=lambda x: x[1] if x[1] is not None else -1,
        reverse=True,
    )[:10]

    return {
        "evidence_type": evidence_type,
        "effect_note": effect_note,
        "preprocessed_effects": effect_dict,
        "preprocessed_p_values": p_value_dict,
        "absolute_effects_preprocessed": absolute_effect_dict,
        "most_influential_preprocessed_feature": most_influential_preprocessed_feature,
        "most_influential_preprocessed_value": (
            absolute_effect_dict.get(most_influential_preprocessed_feature)
            if most_influential_preprocessed_feature
            else None
        ),
        "top_10_preprocessed_features_by_absolute_effect": [
            {
                "feature": feature,
                "absolute_effect": value,
                "effect": effect_dict.get(feature),
                "p_value": p_value_dict.get(feature),
            }
            for feature, value in top_10_preprocessed_features
        ],
        "effect_values": effect_values,
    }


# =========================================================
# 10. Classification evidence
# =========================================================

def get_classification_effects(
    trained_model: Any,
    model_name: str,
    raw_feature_names: List[str],
    cleaned_feature_names: List[str],
    classes: List[Any],
) -> Dict[str, Any]:
    n_features = len(cleaned_feature_names)

    if hasattr(trained_model, "coef_"):
        coef_matrix = np.asarray(trained_model.coef_, dtype=float)

        if coef_matrix.ndim == 1:
            coef_matrix = coef_matrix.reshape(1, -1)

        if coef_matrix.shape[1] < n_features:
            coef_matrix = np.pad(
                coef_matrix,
                ((0, 0), (0, n_features - coef_matrix.shape[1])),
            )
        elif coef_matrix.shape[1] > n_features:
            coef_matrix = coef_matrix[:, :n_features]

        if coef_matrix.shape[0] == 1:
            class_labels = [str(classes[-1]) if classes else "positive_class"]
        else:
            class_labels = [str(c) for c in classes[:coef_matrix.shape[0]]]

        class_specific_effects = {
            class_labels[class_index]: {
                str(feature): safe_float(coef_matrix[class_index][feature_index])
                for feature_index, feature in enumerate(cleaned_feature_names)
            }
            for class_index in range(coef_matrix.shape[0])
        }

        average_absolute_effects = []
        for feature_index in range(n_features):
            average_absolute_effects.append(
                float(np.mean(np.abs(coef_matrix[:, feature_index])))
            )

        evidence_type = "coefficients"
        effect_note = (
            "The model exposes class-specific coefficients. "
            "For multiclass settings, average absolute coefficient magnitude is used "
            "as the overall feature-impact measure."
        )

    elif hasattr(trained_model, "feature_importances_"):
        importances = np.asarray(trained_model.feature_importances_, dtype=float).ravel()

        if len(importances) < n_features:
            importances = np.pad(importances, (0, n_features - len(importances)))
        elif len(importances) > n_features:
            importances = importances[:n_features]

        class_specific_effects = None
        average_absolute_effects = [float(v) for v in importances]

        evidence_type = "feature_importances"
        effect_note = (
            "The model exposes feature importances. "
            "Feature importance indicates relative predictive strength but not positive or negative direction."
        )

    else:
        class_specific_effects = None
        average_absolute_effects = [0.0 for _ in cleaned_feature_names]

        evidence_type = "unavailable"
        effect_note = (
            "The configured model does not expose coefficients or feature importances."
        )

    average_absolute_effect_dict = {
        str(feature): safe_float(value)
        for feature, value in zip(cleaned_feature_names, average_absolute_effects)
    }

    most_influential_preprocessed_feature = (
        max(average_absolute_effect_dict, key=average_absolute_effect_dict.get)
        if average_absolute_effect_dict
        else None
    )

    top_10_preprocessed_features = sorted(
        average_absolute_effect_dict.items(),
        key=lambda x: x[1] if x[1] is not None else -1,
        reverse=True,
    )[:10]

    return {
        "evidence_type": evidence_type,
        "effect_note": effect_note,
        "class_specific_effects": class_specific_effects,
        "average_absolute_effects_preprocessed": average_absolute_effect_dict,
        "most_influential_preprocessed_feature": most_influential_preprocessed_feature,
        "most_influential_preprocessed_value": (
            average_absolute_effect_dict.get(most_influential_preprocessed_feature)
            if most_influential_preprocessed_feature
            else None
        ),
        "top_10_preprocessed_features_by_average_absolute_effect": [
            {
                "feature": feature,
                "average_absolute_effect": value,
            }
            for feature, value in top_10_preprocessed_features
        ],
        "average_absolute_effect_values": average_absolute_effects,
    }


# =========================================================
# 11. Unified configured analysis
# =========================================================

def run_regression_analysis(config: Dict[str, Any]) -> Dict[str, Any]:
    dataset_name = config["dataset_name"]
    x_columns = config["x_columns"]
    y_column = config["y_column"]
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    if task_type != "regression":
        raise ValueError(
            f"This function is for regression experiments only. Received task_type={task_type}."
        )

    df = load_dataset(config)
    validate_columns(df, x_columns, y_column, dataset_name)

    df = df[x_columns + [y_column]].copy()
    df = df.dropna(subset=[y_column])

    X = df[x_columns].copy()
    y_raw = df[y_column].copy()

    y_numeric = pd.to_numeric(y_raw, errors="coerce")
    valid_mask = y_numeric.notna()

    X = X.loc[valid_mask].copy()
    y = y_numeric.loc[valid_mask].copy()

    pipeline, feature_types = build_unified_pipeline(
        X=X,
        x_columns=x_columns,
        model_name=model_name,
        task_type=task_type,
    )

    pipeline.fit(X, y)

    y_pred = pipeline.predict(X)

    full_data_r2 = safe_float(r2_score(y, y_pred))
    full_data_mse = safe_float(mean_squared_error(y, y_pred))
    full_data_rmse = safe_float(math.sqrt(full_data_mse)) if full_data_mse is not None else None
    full_data_mae = safe_float(mean_absolute_error(y, y_pred))

    raw_feature_names, cleaned_feature_names = get_feature_names_from_pipeline(
        pipeline=pipeline,
        feature_types=feature_types,
    )

    numeric_standardized = detect_numeric_standardization(pipeline)

    feature_interpretation_metadata = build_feature_interpretation_metadata(
        raw_feature_names=raw_feature_names,
        cleaned_feature_names=cleaned_feature_names,
        feature_types=feature_types,
        numeric_standardized=numeric_standardized,
    )

    train_test_metrics = compute_train_test_regression_metrics(
        base_pipeline=pipeline,
        X=X,
        y=y,
    )

    effect_summary = get_regression_effects(
        pipeline=pipeline,
        X=X,
        y=y,
        model_name=model_name,
        raw_feature_names=raw_feature_names,
        cleaned_feature_names=cleaned_feature_names,
    )

    original_feature_summary = aggregate_effects_to_original_variables(
        raw_feature_names=raw_feature_names,
        cleaned_feature_names=cleaned_feature_names,
        effect_values=effect_summary["effect_values"],
        numeric_columns=feature_types["numeric_columns"],
        categorical_columns=feature_types["categorical_columns"],
    )

    top_10_original_features = sorted(
        original_feature_summary["original_feature_absolute_effects"].items(),
        key=lambda x: x[1] if x[1] is not None else -1,
        reverse=True,
    )[:10]

    if numeric_standardized:
        coefficient_interpretation_note = (
            "Numeric predictors are median-imputed and standardized using StandardScaler before model fitting. "
            "Therefore, coefficients for numeric predictors should be interpreted as effects of one-standard-deviation "
            "increases after preprocessing, not as one-unit increases in the original raw variables. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded; their coefficients refer to changes "
            "in the encoded category indicators."
        )
        coefficient_interpretation_mode = "standardized_numeric_predictors"
    else:
        coefficient_interpretation_note = (
            "Numeric predictors are not standardized before model fitting. "
            "Therefore, coefficients for numeric predictors can be interpreted as effects of one-unit increases "
            "on the original scale, holding other variables fixed. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded; their coefficients refer to changes "
            "in the encoded category indicators."
        )
        coefficient_interpretation_mode = "raw_numeric_predictors"

    return {
        "task_type": task_type,
        "model_name": model_name,
        "model_display_name": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "dataset_name": dataset_name,
        "file_name": config.get("file_name"),
        "shape_after_dropping_missing_y": list(df.shape),
        "shape_after_valid_numeric_y_filter": [int(X.shape[0]), int(X.shape[1] + 1)],
        "x_columns": x_columns,
        "y_column": y_column,
        "numeric_columns": feature_types["numeric_columns"],
        "categorical_columns": feature_types["categorical_columns"],
        "preprocessed_feature_names": cleaned_feature_names,
        "feature_interpretation_metadata": feature_interpretation_metadata,
        "performance": {
            "r2_full_data": full_data_r2,
            "rmse_full_data": full_data_rmse,
            "mae_full_data": full_data_mae,
            "train_r2": train_test_metrics["train_r2"],
            "test_r2": train_test_metrics["test_r2"],
            "train_rmse": train_test_metrics["train_rmse"],
            "test_rmse": train_test_metrics["test_rmse"],
            "train_mae": train_test_metrics["train_mae"],
            "test_mae": train_test_metrics["test_mae"],
            "metric_note": "The full-data metrics are calculated on the same data used for fitting.",
            "split_note": train_test_metrics["split_note"],
        },
        "evidence_type": effect_summary["evidence_type"],
        "effect_note": effect_summary["effect_note"],
        "preprocessed_effects": effect_summary["preprocessed_effects"],
        "preprocessed_p_values": effect_summary["preprocessed_p_values"],
        "absolute_effects_preprocessed": effect_summary["absolute_effects_preprocessed"],
        "most_influential_preprocessed_feature": effect_summary["most_influential_preprocessed_feature"],
        "most_influential_preprocessed_value": effect_summary["most_influential_preprocessed_value"],
        "original_feature_summary": original_feature_summary,
        "top_10_preprocessed_features_by_absolute_effect": (
            effect_summary["top_10_preprocessed_features_by_absolute_effect"]
        ),
        "top_10_original_features_by_absolute_effect": [
            {
                "feature": feature,
                "absolute_effect": value,
                "representative_effect": (
                    original_feature_summary["original_feature_representative_effects"].get(feature)
                ),
            }
            for feature, value in top_10_original_features
        ],
        "numeric_standardized": numeric_standardized,
        "coefficient_interpretation_mode": coefficient_interpretation_mode,
        "coefficient_interpretation_note": coefficient_interpretation_note,
        "preprocessing_note": coefficient_interpretation_note,
    }


def run_classification_analysis(config: Dict[str, Any]) -> Dict[str, Any]:
    dataset_name = config["dataset_name"]
    x_columns = config["x_columns"]
    y_column = config["y_column"]
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    if task_type not in ["binary_classification", "multiclass_classification"]:
        raise ValueError(
            f"This function is for classification experiments only. Received task_type={task_type}."
        )

    df = load_dataset(config)
    validate_columns(df, x_columns, y_column, dataset_name)

    df = df[x_columns + [y_column]].copy()
    df = df.dropna(subset=[y_column])

    X = df[x_columns].copy()
    y = df[y_column].copy()

    pipeline, feature_types = build_unified_pipeline(
        X=X,
        x_columns=x_columns,
        model_name=model_name,
        task_type=task_type,
    )

    pipeline.fit(X, y)

    y_pred = pipeline.predict(X)
    full_data_accuracy = safe_float(accuracy_score(y, y_pred))

    raw_feature_names, cleaned_feature_names = get_feature_names_from_pipeline(
        pipeline=pipeline,
        feature_types=feature_types,
    )

    numeric_standardized = detect_numeric_standardization(pipeline)

    feature_interpretation_metadata = build_feature_interpretation_metadata(
        raw_feature_names=raw_feature_names,
        cleaned_feature_names=cleaned_feature_names,
        feature_types=feature_types,
        numeric_standardized=numeric_standardized,
    )

    trained_model = pipeline.named_steps["model"]

    classes = list(getattr(trained_model, "classes_", sorted(pd.Series(y).dropna().unique())))
    class_distribution = y.value_counts(dropna=False).to_dict()
    class_distribution = {
        str(k): int(v)
        for k, v in class_distribution.items()
    }

    train_test_accuracy = compute_train_test_accuracy(
        base_pipeline=pipeline,
        X=X,
        y=y,
    )

    effect_summary = get_classification_effects(
        trained_model=trained_model,
        model_name=model_name,
        raw_feature_names=raw_feature_names,
        cleaned_feature_names=cleaned_feature_names,
        classes=classes,
    )

    original_feature_summary = aggregate_effects_to_original_variables(
        raw_feature_names=raw_feature_names,
        cleaned_feature_names=cleaned_feature_names,
        effect_values=effect_summary["average_absolute_effect_values"],
        numeric_columns=feature_types["numeric_columns"],
        categorical_columns=feature_types["categorical_columns"],
    )

    top_10_original_features = sorted(
        original_feature_summary["original_feature_absolute_effects"].items(),
        key=lambda x: x[1] if x[1] is not None else -1,
        reverse=True,
    )[:10]

    return {
        "task_type": task_type,
        "model_name": model_name,
        "model_display_name": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "dataset_name": dataset_name,
        "file_name": config.get("file_name"),
        "shape_after_dropping_missing_y": list(df.shape),
        "x_columns": x_columns,
        "y_column": y_column,
        "numeric_columns": feature_types["numeric_columns"],
        "categorical_columns": feature_types["categorical_columns"],
        "classes": [str(c) for c in classes],
        "num_classes": int(len(classes)),
        "class_distribution": class_distribution,
        "preprocessed_feature_names": cleaned_feature_names,
        "feature_interpretation_metadata": feature_interpretation_metadata,
        "performance": {
            "accuracy_full_data": full_data_accuracy,
            "train_accuracy": train_test_accuracy["train_accuracy"],
            "test_accuracy": train_test_accuracy["test_accuracy"],
            "metric_note": "The full-data accuracy is calculated on the same data used for fitting.",
            "split_note": train_test_accuracy["split_note"],
        },
        "evidence_type": effect_summary["evidence_type"],
        "effect_note": effect_summary["effect_note"],
        "class_specific_effects": effect_summary["class_specific_effects"],
        "average_absolute_effects_preprocessed": effect_summary["average_absolute_effects_preprocessed"],
        "most_influential_preprocessed_feature": effect_summary["most_influential_preprocessed_feature"],
        "most_influential_preprocessed_value": effect_summary["most_influential_preprocessed_value"],
        "original_feature_summary": original_feature_summary,
        "top_10_preprocessed_features_by_average_absolute_effect": (
            effect_summary["top_10_preprocessed_features_by_average_absolute_effect"]
        ),
        "top_10_original_features_by_average_absolute_effect": [
            {
                "feature": feature,
                "average_absolute_effect": value,
                "representative_effect": (
                    original_feature_summary["original_feature_representative_effects"].get(feature)
                ),
            }
            for feature, value in top_10_original_features
        ],
        "significance_note": (
            "P-values are not computed for the configured classification models in this experiment. "
            "For classification, coefficient magnitude or feature importance is used as the feature-impact evidence."
        ),
        "numeric_standardized": numeric_standardized,
        "preprocessing_note": (
            "Numeric predictors are median-imputed and standardized. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded. "
            "Coefficients or feature importances correspond to preprocessed model features. "
            "Original-feature importance aggregates one-hot encoded features back to their source variables using maximum absolute effect."
        ),
    }


def run_configured_analysis_backend(config: Dict[str, Any]) -> Dict[str, Any]:
    task_type = infer_task_type(config)

    if task_type == "regression":
        return run_regression_analysis(config)

    if task_type in ["binary_classification", "multiclass_classification"]:
        return run_classification_analysis(config)

    raise ValueError(f"Unsupported task type: {task_type}")


# =========================================================
# 12. Token and runtime extraction
# =========================================================

def normalize_usage(usage: Optional[Dict[str, Any]]) -> Dict[str, int]:
    if not usage:
        return {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        }

    prompt_tokens = (
        usage.get("prompt_tokens")
        or usage.get("input_tokens")
        or usage.get("input_token_count")
        or 0
    )

    completion_tokens = (
        usage.get("completion_tokens")
        or usage.get("output_tokens")
        or usage.get("output_token_count")
        or 0
    )

    total_tokens = (
        usage.get("total_tokens")
        or usage.get("total_token_count")
        or 0
    )

    if not total_tokens:
        total_tokens = prompt_tokens + completion_tokens

    return {
        "prompt_tokens": int(prompt_tokens),
        "completion_tokens": int(completion_tokens),
        "total_tokens": int(total_tokens),
    }


def extract_message_usage(message: Any) -> Dict[str, int]:
    metadata = getattr(message, "response_metadata", None)

    if isinstance(metadata, dict):
        token_usage = metadata.get("token_usage")
        if token_usage:
            return normalize_usage(token_usage)

        usage = metadata.get("usage")
        if usage:
            return normalize_usage(usage)

    usage_metadata = getattr(message, "usage_metadata", None)

    if isinstance(usage_metadata, dict):
        return normalize_usage(usage_metadata)

    return normalize_usage(None)


def summarize_agent_run(result: Dict[str, Any], runtime_seconds: float) -> Dict[str, Any]:
    messages = result.get("messages", [])

    prompt_tokens = 0
    completion_tokens = 0
    total_tokens = 0
    llm_call_count = 0
    tool_call_count = 0

    for message in messages:
        usage = extract_message_usage(message)

        if usage["total_tokens"] > 0:
            llm_call_count += 1
            prompt_tokens += usage["prompt_tokens"]
            completion_tokens += usage["completion_tokens"]
            total_tokens += usage["total_tokens"]

        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            tool_call_count += len(tool_calls)

    return {
        "llm_call_count": llm_call_count,
        "tool_call_count": tool_call_count,
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
        },
        "runtime_seconds": float(runtime_seconds),
    }


def serialize_message(message: Any) -> Dict[str, Any]:
    message_type = getattr(message, "type", None) or message.__class__.__name__
    content = getattr(message, "content", "")

    serialized = {
        "type": message_type,
        "content": content,
    }

    name = getattr(message, "name", None)
    if name:
        serialized["name"] = name

    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        serialized["tool_calls"] = make_json_safe(tool_calls)

    usage = extract_message_usage(message)
    if usage["total_tokens"] > 0:
        serialized["usage"] = usage

    response_metadata = getattr(message, "response_metadata", None)
    if response_metadata:
        serialized["response_metadata"] = make_json_safe(response_metadata)

    return serialized


# =========================================================
# 13. Build LangChain agent for GPT-5.5
# =========================================================

class InspectDatasetInput(BaseModel):
    dataset_path: str = Field(description="Path to the CSV dataset file.")


class RunConfiguredAnalysisInput(BaseModel):
    dataset_path: str = Field(description="Path to the CSV dataset file.")


def build_gpt55_llm() -> ChatOpenAI:
    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError("Please set OPENAI_API_KEY.")

    kwargs = {
        "model": OPENAI_MODEL,
        "api_key": os.environ["OPENAI_API_KEY"],
    }

    # Do not include temperature or other sampling parameters for GPT-5.5.
    return ChatOpenAI(**kwargs)


def build_agent(config: Dict[str, Any]):
    llm = build_gpt55_llm()

    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    @tool(args_schema=InspectDatasetInput)
    def inspect_dataset(dataset_path: str) -> str:
        """Inspect the dataset and return its shape, columns, and data types."""
        df = load_dataset(config)

        result = {
            "shape": list(df.shape),
            "columns": list(df.columns),
            "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
            "dataset_path_received": dataset_path,
            "dataset_path_used": get_dataset_path(config),
        }

        return json.dumps(make_json_safe(result), ensure_ascii=False, indent=2)

    @tool(args_schema=RunConfiguredAnalysisInput)
    def run_configured_analysis(dataset_path: str) -> str:
        """
        Run the configured analysis for the current experiment.

        This tool supports regression, binary classification, and multiclass classification.
        """
        result = run_configured_analysis_backend(config)
        return json.dumps(make_json_safe(result), ensure_ascii=False, indent=2)

    tools = [
        inspect_dataset,
        run_configured_analysis,
    ]

    system_prompt = f"""
You are a LangChain-based tool-using single-agent baseline for analytical textual reporting.

Your task is to answer one user question at a time about a structured dataset.

Current experiment settings:
- Task type: {task_type}
- X columns: {config["x_columns"]}
- Y column: {config["y_column"]}
- Configured model for this experiment: {model_name}
- Configured model display name: {MODEL_DISPLAY_NAMES.get(model_name, model_name)}
- LLM provider: {LLM_PROVIDER}
- LLM model: {OPENAI_MODEL}

Tool-use rules:
- You must call run_configured_analysis before answering every analytical question.
- Use inspect_dataset only when dataset structure, columns, or data types need to be checked.
- Do not answer from prior knowledge or assumptions.
- Base the final answer on the tool output.
- If the task is regression, discuss regression metrics such as R-squared, RMSE, MAE, coefficients, p-values if available, or feature importance.
- If the task is classification, discuss accuracy, class distribution, coefficients, or feature importance.
- When numeric predictors are standardized, avoid describing numeric coefficients as raw one-unit effects. Use the preprocessing and interpretation notes returned by the tool.
- Keep the answer focused on the user's question.
""".strip()

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt,
    )

    return agent


# =========================================================
# 14. Agent query
# =========================================================

def ask_agent(
    agent: Any,
    config: Dict[str, Any],
    question: str,
) -> Dict[str, Any]:
    dataset_path = get_dataset_path(config)
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    prompt = f"""
Dataset path:
{dataset_path}

Dataset name:
{config["dataset_name"]}

Task type:
{task_type}

Configured model:
{model_name}

Background information:
{config["background_knowledge"]}

Question:
{question}

Please answer the question using the available tools.
You must call run_configured_analysis before producing the final answer.
""".strip()

    start_time = time.time()

    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    runtime_seconds = time.time() - start_time

    messages = result.get("messages", [])
    final_message = messages[-1] if messages else None
    answer = getattr(final_message, "content", str(final_message)) if final_message else ""

    run_summary = summarize_agent_run(result, runtime_seconds)

    output = {
        "question": question,
        "answer": answer,
        "status": "success" if answer else "failure",
        "failure_reason": None if answer else "Empty final answer.",
        "llm_call_count": run_summary["llm_call_count"],
        "tool_call_count": run_summary["tool_call_count"],
        "usage": run_summary["usage"],
        "runtime_seconds": run_summary["runtime_seconds"],
    }

    if SAVE_MESSAGE_TRACE:
        output["messages_trace"] = [serialize_message(m) for m in messages]

    return output


# =========================================================
# 15. Summary helpers
# =========================================================

def summarize_dataset_output(output: Dict[str, Any]) -> Dict[str, Any]:
    total_llm_call_count = 0
    total_tool_call_count = 0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0
    total_runtime_seconds = 0.0
    success_count = 0
    failure_count = 0

    for qa in output["questions_and_answers"]:
        if qa.get("status") == "success":
            success_count += 1
        else:
            failure_count += 1

        total_llm_call_count += qa.get("llm_call_count", 0) or 0
        total_tool_call_count += qa.get("tool_call_count", 0) or 0

        usage = qa.get("usage") or {}

        total_prompt_tokens += usage.get("prompt_tokens", 0) or 0
        total_completion_tokens += usage.get("completion_tokens", 0) or 0
        total_tokens += usage.get("total_tokens", 0) or 0

        total_runtime_seconds += qa.get("runtime_seconds", 0.0) or 0.0

    n = len(output["questions_and_answers"])

    return {
        "num_questions": n,
        "success_count": success_count,
        "failure_count": failure_count,
        "failure_rate": failure_count / n if n else None,
        "total_llm_call_count": total_llm_call_count,
        "total_tool_call_count": total_tool_call_count,
        "total_prompt_tokens": total_prompt_tokens,
        "total_completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
        "total_runtime_seconds": total_runtime_seconds,
        "avg_llm_call_count_per_answer": total_llm_call_count / n if n else None,
        "avg_tool_call_count_per_answer": total_tool_call_count / n if n else None,
        "avg_total_tokens_per_answer": total_tokens / n if n else None,
        "avg_runtime_seconds_per_answer": total_runtime_seconds / n if n else None,
    }


def summarize_all_outputs(all_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    total_questions = 0
    total_failures = 0
    total_llm_calls = 0
    total_tool_calls = 0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0
    total_runtime = 0.0

    for output in all_outputs:
        summary = output.get("dataset_summary") or {}

        total_questions += summary.get("num_questions", 0) or 0
        total_failures += summary.get("failure_count", 0) or 0
        total_llm_calls += summary.get("total_llm_call_count", 0) or 0
        total_tool_calls += summary.get("total_tool_call_count", 0) or 0
        total_prompt_tokens += summary.get("total_prompt_tokens", 0) or 0
        total_completion_tokens += summary.get("total_completion_tokens", 0) or 0
        total_tokens += summary.get("total_tokens", 0) or 0
        total_runtime += summary.get("total_runtime_seconds", 0.0) or 0.0

    return {
        "num_datasets": len(all_outputs),
        "total_questions": total_questions,
        "total_failures": total_failures,
        "overall_failure_rate": total_failures / total_questions if total_questions else None,
        "total_llm_call_count": total_llm_calls,
        "total_tool_call_count": total_tool_calls,
        "total_prompt_tokens": total_prompt_tokens,
        "total_completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
        "total_runtime_seconds": total_runtime,
        "avg_llm_call_count_per_answer": total_llm_calls / total_questions if total_questions else None,
        "avg_tool_call_count_per_answer": total_tool_calls / total_questions if total_questions else None,
        "avg_total_tokens_per_answer": total_tokens / total_questions if total_questions else None,
        "avg_runtime_seconds_per_answer": total_runtime / total_questions if total_questions else None,
    }


# =========================================================
# 16. Run one dataset
# =========================================================

def make_failure_qa(question: str, failure_reason: str) -> Dict[str, Any]:
    return {
        "question": question,
        "answer": None,
        "status": "failure",
        "failure_reason": failure_reason,
        "llm_call_count": 0,
        "tool_call_count": 0,
        "usage": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        },
        "runtime_seconds": 0.0,
    }


def run_one_experiment(config: Dict[str, Any]) -> Dict[str, Any]:
    dataset_name = config["dataset_name"]
    dataset_path = get_dataset_path(config)
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    output = {
        "baseline_name": "langchain_tool_using_single_agent",
        "llm_provider": LLM_PROVIDER,
        "model_display_name": OPENAI_MODEL_DISPLAY_NAME,
        "model_id": OPENAI_MODEL,
        **get_sampling_control_metadata(),
        "cost_measurement_mode": "per_question_isolated_agent_run",
        "dataset_name": dataset_name,
        "dataset_path": dataset_path,
        "file_name": config.get("file_name"),
        "background_knowledge": config["background_knowledge"],
        "task_type": task_type,
        "configured_model_name": model_name,
        "configured_model_display_name": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "x_columns": config["x_columns"],
        "y_column": config["y_column"],
        "questions_and_answers": [],
    }

    if not os.path.exists(dataset_path):
        for question in config["questions"]:
            output["questions_and_answers"].append(
                make_failure_qa(question, f"Dataset not found: {dataset_path}")
            )

        output["dataset_summary"] = summarize_dataset_output(output)
        return output

    try:
        agent = build_agent(config)

        for question in config["questions"]:
            try:
                qa_output = ask_agent(agent, config, question)

            except Exception as exc:
                qa_output = make_failure_qa(question, str(exc))

            output["questions_and_answers"].append(qa_output)

    except Exception as exc:
        for question in config["questions"]:
            output["questions_and_answers"].append(
                make_failure_qa(question, f"Agent construction failed: {str(exc)}")
            )

    output["dataset_summary"] = summarize_dataset_output(output)
    return output


# =========================================================
# 17. Main
# =========================================================

def main():
    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError("Please set OPENAI_API_KEY.")

    if os.environ.get("OPENAI_API_KEY") == "your_openai_api_key_here":
        raise EnvironmentError("Please replace the placeholder OpenAI API key before running.")

    configs = load_experiment_configs(CONFIG_PATH)
    all_outputs = []

    config_stem = safe_name(os.path.splitext(os.path.basename(CONFIG_PATH))[0])

    for config in configs:
        dataset_name = config["dataset_name"]
        model_name = get_model_name(config)
        task_type = infer_task_type(config)

        print("=" * 100)
        print(f"Running dataset: {dataset_name}")
        print(f"Task type: {task_type}")
        print(f"Configured model: {model_name}")
        print(f"LLM provider: {LLM_PROVIDER}")
        print(f"LLM model: {OPENAI_MODEL}")
        print("Temperature sent: False")
        print("=" * 100)

        result = run_one_experiment(config)
        all_outputs.append(result)

        dataset_output_path = os.path.join(
            OUTPUT_DIR,
            f"{safe_name(dataset_name)}_{safe_name(model_name)}_langchain_agent_gpt55_outputs.json"
        )

        with open(dataset_output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print("Dataset summary:")
        print(json.dumps(result["dataset_summary"], ensure_ascii=False, indent=2))

        for qa in result["questions_and_answers"]:
            print("-" * 80)
            print(qa["question"])
            print("Status:", qa["status"])
            print("Failure reason:", qa.get("failure_reason"))
            print("LLM calls:", qa["llm_call_count"])
            print("Tool calls:", qa["tool_call_count"])
            print("Prompt tokens:", qa["usage"]["prompt_tokens"])
            print("Completion tokens:", qa["usage"]["completion_tokens"])
            print("Total tokens:", qa["usage"]["total_tokens"])
            print("Runtime seconds:", qa["runtime_seconds"])
            print("Answer:")
            print(qa["answer"])
            print()

        print(f"Saved dataset output to: {dataset_output_path}")

    combined_output = {
        "baseline_name": "langchain_tool_using_single_agent",
        "llm_provider": LLM_PROVIDER,
        "model_display_name": OPENAI_MODEL_DISPLAY_NAME,
        "model_id": OPENAI_MODEL,
        **get_sampling_control_metadata(),
        "config_path": CONFIG_PATH,
        "outputs": all_outputs,
        "overall_summary": summarize_all_outputs(all_outputs),
    }

    combined_output_path = os.path.join(
        OUTPUT_DIR,
        f"all_{config_stem}_langchain_agent_gpt55_outputs.json"
    )

    with open(combined_output_path, "w", encoding="utf-8") as f:
        json.dump(combined_output, f, ensure_ascii=False, indent=2)

    print("=" * 100)
    print("Overall summary:")
    print(json.dumps(combined_output["overall_summary"], ensure_ascii=False, indent=2))
    print(f"Saved combined output to: {combined_output_path}")


main()